<a href="https://colab.research.google.com/github/chiemahp/Flyrank-internship-ml/blob/main/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chiemahp/Flyrank-internship-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

repo_root = Path.cwd()
raw_path = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"
for parent in [repo_root, *repo_root.parents]:
    candidate = parent / "data" / "raw" / "content_refresh_anonymized.csv"
    if candidate.exists():
        raw_path = candidate
        repo_root = parent
        break

df = pd.read_csv(raw_path)

numeric_cols = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "ai_sessions_90d",
    "scroll_events_90d", "days_with_impressions", "days_with_sessions",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_age_days", "age_tier_order", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate",
    "ai_traffic_pct", "trend_pct",
]
categorical_cols = [
    "competition_level", "content_type", "main_intent", "provider_used",
    "model_used", "age_tier", "freshness_tier", "word_count_tier",
    "char_count_tier", "impression_tier", "position_tier", "trend_direction",
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

for col in categorical_cols:
    df[col] = df[col].fillna("unknown").astype(str).replace({"": "unknown", "nan": "unknown"})

df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()

df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])
df["has_clicks"] = (df["clicks_90d"] > 0).astype(int)
df["has_ai_sessions"] = (df["ai_sessions_90d"] > 0).astype(int)
df["measurable_opportunity"] = ((df["impressions_90d"] >= 100) & (df["sessions_90d"] > 0)).astype(int)

model_numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d",
    "log_ai_sessions_90d", "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update", "ctr", "avg_position",
    "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
model_categorical_features = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]

feature_frame = df[["content_id", "client_id", "is_declining_label"] + model_numeric_features + model_categorical_features].copy()
for col in model_categorical_features:
    feature_frame[col] = feature_frame[col].astype("category")

print(f"Prepared rows: {len(feature_frame):,}")
feature_frame.head()

Prepared rows: 30,000


,content_id,client_id,is_declining_label,search_volume,competition,cpc,word_count,char_count,log_impressions_90d,log_clicks_90d,...,scroll_rate,ai_traffic_pct,competition_level,content_type,main_intent,age_tier,freshness_tier,word_count_tier,impression_tier,position_tier
0,content_304f48230142,client_f369cb89fc,1,10.0,0.67,2.05,3221.0,20457.0,8.243808,3.401197,...,4.55,0.0,HIGH,keyword article,transactional,181-365,0-30,2000-3500,good,striking
1,content_a1fb4e703a9e,client_4e07408562,1,90.0,0.01,0.05,2481.0,15562.0,9.636980,2.079442,...,10.00,0.0,LOW,keyword article,informational,365+,0-30,2000-3500,good,page_3_5
2,content_9aa793d4d895,client_7f2253d7e2,1,0.0,0.00,0.00,3515.0,23643.0,9.440023,2.484907,...,28.57,0.0,LOW,keyword article,informational,91-180,0-30,3500+,good,page_3_5
3,content_331d6c4de07b,client_19581e27de,0,10.0,0.00,0.00,0.0,0.0,9.371779,4.077537,...,3.45,0.0,LOW,keyword article,commercial,365+,0-30,unknown,good,page_1
4,content_d99b7a2d90ca,client_3fdba35f04,1,0.0,0.00,0.00,2803.0,17469.0,9.859588,3.218876,...,24.29,0.0,LOW,keyword article,informational,181-365,0-30,2000-3500,good,page_3_5


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [ ]:
# Feature notes based on the repository's data dictionary and the prepared frame.
feature_notes = []
for col in model_numeric_features + model_categorical_features:
    if col in feature_frame.columns:
        if col in model_numeric_features:
            missing_count = int(feature_frame[col].eq(0).sum())
            note = f"{col}: numeric feature; zeros are used for missing/blank values after prep, and the column is available at prediction time."
        else:
            missing_count = int(feature_frame[col].eq("unknown").sum())
            note = f"{col}: categorical feature; missing values are filled as 'unknown', so the category is known at prediction time."
        feature_notes.append((col, missing_count, note))

feature_notes[:10], len(feature_notes)

([('search_volume',
   13549,
   'search_volume: numeric feature; zeros are used for missing/blank values after prep, and the column is available at prediction time.'),
  ('competition',
   18847,
   'competition: numeric feature; zeros are used for missing/blank values after prep, and the column is available at prediction time.'),
  ('cpc',
   23147,
   'cpc: numeric feature; zeros are used for missing/blank values after prep, and the column is available at prediction time.'),
  ('word_count',
   7699,
   'word_count: numeric feature; zeros are used for missing/blank values after prep, and the column is available at prediction time.'),
  ('char_count',
   7699,
   'char_count: numeric feature; zeros are used for missing/blank values after prep, and the column is available at prediction time.'),
  ('log_impressions_90d',
   0,
   'log_impressions_90d: numeric feature; zeros are used for missing/blank values after prep, and the column is available at prediction time.'),
  ('log_clicks_9

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

# Leakage test 1: compare a model with a suspected label-derived signal to one without it.
leak_candidate = "trend_pct"
feature_cols = [c for c in model_numeric_features + model_categorical_features if c != leak_candidate]

X = feature_frame[feature_cols].copy()
y = feature_frame["is_declining_label"]

numeric_features = [c for c in feature_cols if c in model_numeric_features]
categorical_features = [c for c in feature_cols if c in model_categorical_features]

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_features),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
    ]
)

model = make_pipeline(preprocess, LogisticRegression(max_iter=2000, solver="liblinear"))

cv = GroupKFold(n_splits=3)
cv_scores = []
for train_idx, test_idx in cv.split(X, y, groups=df["client_id"]):
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    prob = model.predict_proba(X.iloc[test_idx])[:, 1]
    cv_scores.append(roc_auc_score(y.iloc[test_idx], prob))

leaky_scores = []
for train_idx, test_idx in cv.split(X, y, groups=df["client_id"]):
    X_leaky = X.copy()
    X_leaky[leak_candidate] = df.loc[X.index, leak_candidate].to_numpy()
    model_leaky = make_pipeline(
        ColumnTransformer(
            transformers=[
                ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_features + [leak_candidate]),
                ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
            ]
        ),
        LogisticRegression(max_iter=2000, solver="liblinear"),
    )
    model_leaky.fit(X_leaky.iloc[train_idx], y.iloc[train_idx])
    prob = model_leaky.predict_proba(X_leaky.iloc[test_idx])[:, 1]
    leaky_scores.append(roc_auc_score(y.iloc[test_idx], prob))

cv_auc = float(np.mean(cv_scores))
leaky_auc = float(np.mean(leaky_scores))

print({
    "honest_cv_auc": round(cv_auc, 4),
    "with_label_source_feature": round(leaky_auc, 4),
    "difference": round(leaky_auc - cv_auc, 4),
    "base_rate": round(y.mean(), 4),
})

{'honest_cv_auc': 0.6378, 'with_label_source_feature': 0.9987, 'difference': 0.3609, 'base_rate': np.float64(0.5421)}


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
excluded_fields = [
    "content_id", "client_id", "trend_direction", "trend_pct", "provider_used", "model_used",
]
exclusion_reasons = {
    "content_id": "Unique identifier; grouping only, never a feature.",
    "client_id": "Grouping key for splits; not a predictive signal.",
    "trend_direction": "This is the target label source and must not be used as a feature.",
    "trend_pct": "Label-derived signal from the same outcome window; leakage risk.",
    "provider_used": "Metadata about the generation process, not an input that should drive a product decision.",
    "model_used": "Model identity is not a business-ready feature for this task.",
}

excluded_fields, exclusion_reasons

(['content_id',
  'client_id',
  'trend_direction',
  'trend_pct',
  'provider_used',
  'model_used'],
 {'content_id': 'Unique identifier; grouping only, never a feature.',
  'client_id': 'Grouping key for splits; not a predictive signal.',
  'trend_direction': 'This is the target label source and must not be used as a feature.',
  'trend_pct': 'Label-derived signal from the same outcome window; leakage risk.',
  'provider_used': 'Metadata about the generation process, not an input that should drive a product decision.',
  'model_used': 'Model identity is not a business-ready feature for this task.'})

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.